# PW2 EX2


In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [4]:
import cv2
import numpy as np

# Importing images
imgR = cv2.imread('./data/train.jpg')
grayR = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)

imgL = cv2.imread("./data/query.jpg")
grayL = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)


sift = cv2.SIFT_create()

# Finding key points — ✅ using grayscale
kpR, dsR = sift.detectAndCompute(grayR, None)
kpL, dsL = sift.detectAndCompute(grayL, None)

cv2.imshow('RIGHT IMAGE KEY POINTS', cv2.drawKeypoints(imgR, kpR, None))
cv2.waitKey(0)

cv2.imshow('LEFT IMAGE KEY POINTS', cv2.drawKeypoints(imgL, kpL, None))
cv2.waitKey(0)


match = cv2.BFMatcher()
matches = match.knnMatch(dsR, dsL, k=2)


# ✅ Lowe's ratio test with 0.75 threshold
good = []
for m, n in matches:
    if m.distance < 0.3 * n.distance:
        good.append(m)


# ✅ Fixed "matchColor" typo
draw_params = dict(matchColor=(0, 255, 0),
                   singlePointColor=None,
                   flags=2)

img3 = cv2.drawMatches(imgR, kpR, imgL, kpL, good, None, **draw_params)
cv2.imshow("IMAGE AFTER DRAWING MATCHES", img3)
cv2.waitKey(0)


MIN_MATCH_COUNT = 10
if len(good) > MIN_MATCH_COUNT:
    src_pts = np.float32([kpR[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kpL[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # ✅ Fixed shape unpacking for color image
    h, w = imgR.shape[:2]
    pts = np.float32([[0, 0], [0, h-1], [w-1, h-1], [w-1, 0]]).reshape(-1, 1, 2)
    dst = cv2.perspectiveTransform(pts, M)

    imgR_drawn = cv2.polylines(imgR.copy(), [np.int32(dst)], True, 255, 3, cv2.LINE_AA)
    cv2.imshow("AFTER POLYLINES", imgR_drawn)
    cv2.waitKey(0)

else:
    print(f"Not enough matches found: {len(good)}/{MIN_MATCH_COUNT}")


# ✅ Canvas size = combined width x height
canvas_width  = imgL.shape[1] + imgR.shape[1]
canvas_height = imgL.shape[0]
print(canvas_width, canvas_height)

# ✅ warpPerspective now receives a proper (width, height) tuple
dst = cv2.warpPerspective(imgR, M, (canvas_width, canvas_height))
cv2.imshow("dst warpPerspective", dst)
cv2.waitKey(0)


def trim(frame):
    # Remove black (zero) borders recursively from all 4 sides
    if not np.sum(frame[0]):          # top row all black
        return trim(frame[1:])

    if not np.sum(frame[-1]):         # bottom row all black
        return trim(frame[0:-1])

    if not np.sum(frame[:, 0]):       # left column all black
        return trim(frame[:, 1:])

    if not np.sum(frame[:, -1]):      # ✅ Fixed :-2 → :-1 (remove only 1 column)
        return trim(frame[:, :-1])

    return frame


# ✅ Added .jpg extension to imwrite
cv2.imshow("STITCHED IMAGE", trim(dst))
cv2.waitKey(0)
cv2.imwrite("STITCHED_IMAGE_CROP.jpg", trim(dst))

cv2.destroyAllWindows()

2922 819
